In [2]:
# ============================================================
# PEARLS AQI PREDICTOR
# THREE-DAY AQI FORECASTING
# ============================================================

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost

print("=" * 60)
print("PEARLS AQI PREDICTOR")
print("THREE-DAY AQI FORECASTING")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

CHAMPION_MODEL = "models:/Pearls_AQI_XGBoost@champion"

FORECAST_HOURS = 72

print("MLflow URI:", MLFLOW_TRACKING_URI)
print("Champion model:", CHAMPION_MODEL)
print("Forecast horizon:", FORECAST_HOURS, "hours")

# ------------------------------------------------------------
# CONNECT TO MLFLOW
# ------------------------------------------------------------

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)

print("\nMLflow connected.")

PEARLS AQI PREDICTOR
THREE-DAY AQI FORECASTING
MLflow URI: http://127.0.0.1:5000
Champion model: models:/Pearls_AQI_XGBoost@champion
Forecast horizon: 72 hours

MLflow connected.


In [ ]:
# ============================================================
# LOAD MLFLOW CHAMPION MODEL
# ============================================================

print("=" * 60)
print("LOADING MLFLOW CHAMPION MODEL")
print("=" * 60)

champion_model = mlflow.xgboost.load_model(
    CHAMPION_MODEL
)

print("Champion model loaded successfully.")
print("Model type:", type(champion_model))

LOADING MLFLOW CHAMPION MODEL


Champion model loaded successfully.
Model type: <class 'xgboost.sklearn.XGBRegressor'>


In [ ]:
# ============================================================
# VERIFY CHAMPION MODEL FEATURES
# ============================================================

print("=" * 60)
print("CHAMPION MODEL FEATURE VERIFICATION")
print("=" * 60)

# XGBoost stores the feature names it was trained with
model_features = champion_model.get_booster().feature_names

print("Number of features:", len(model_features))

print("\nFeatures expected by champion model:")
for i, feature in enumerate(model_features, start=1):
    print(f"{i:2}. {feature}")

print("\n" + "=" * 60)

if len(model_features) == 70:
    print("✓ Champion model expects exactly 70 features")
else:
    print(
        "⚠ Unexpected feature count:",
        len(model_features)
    )

CHAMPION MODEL FEATURE VERIFICATION
Number of features: 70

Features expected by champion model:
 1. temperature_2m
 2. relative_humidity_2m
 3. pressure_msl
 4. precipitation
 5. wind_speed_10m
 6. wind_direction_10m
 7. pm2_5
 8. pm10
 9. carbon_monoxide
10. nitrogen_dioxide
11. sulphur_dioxide
12. ozone
13. hour
14. day_of_week
15. day_of_month
16. month
17. is_weekend
18. aqi_lag_1
19. aqi_lag_3
20. aqi_lag_6
21. aqi_lag_12
22. aqi_lag_24
23. aqi_lag_48
24. aqi_lag_72
25. pm2_5_lag_1
26. pm2_5_lag_3
27. pm2_5_lag_6
28. pm2_5_lag_24
29. pm10_lag_1
30. pm10_lag_3
31. pm10_lag_6
32. pm10_lag_24
33. carbon_monoxide_lag_1
34. carbon_monoxide_lag_3
35. carbon_monoxide_lag_6
36. carbon_monoxide_lag_24
37. nitrogen_dioxide_lag_1
38. nitrogen_dioxide_lag_3
39. nitrogen_dioxide_lag_6
40. nitrogen_dioxide_lag_24
41. sulphur_dioxide_lag_1
42. sulphur_dioxide_lag_3
43. sulphur_dioxide_lag_6
44. sulphur_dioxide_lag_24
45. ozone_lag_1
46. ozone_lag_3
47. ozone_lag_6
48. ozone_lag_24
49. aqi_3h_me

In [9]:
# ============================================================
#  LOAD PROCESSED AQI FEATURES
# ============================================================

from pathlib import Path

DATA_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "aqi_features.parquet"
)

print("=" * 60)
print("LOADING PROCESSED AQI FEATURES")
print("=" * 60)

print("Path:", DATA_PATH)

df_forecast = pd.read_parquet(DATA_PATH)

# ------------------------------------------------------------
# TIMESTAMP
# ------------------------------------------------------------

df_forecast["timestamp"] = pd.to_datetime(
    df_forecast["timestamp"],
    utc=True
)

df_forecast = (
    df_forecast
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("\nDataset loaded successfully.")
print("Shape:", df_forecast.shape)

print("\nDate range:")
print(
    df_forecast["timestamp"].min(),
    "→",
    df_forecast["timestamp"].max()
)

print("\nColumns:", len(df_forecast.columns))

print("\nLatest available record:")
display(
    df_forecast[
        [
            "timestamp",
            "us_aqi",
            "pm2_5",
            "pm10",
            "ozone"
        ]
    ].tail(1)
)

LOADING PROCESSED AQI FEATURES
Path: d:\Internship\pearls-aqi-predictor\data\processed\aqi_features.parquet

Dataset loaded successfully.
Shape: (17471, 82)

Date range:
2024-08-04 00:00:00+00:00 → 2026-08-01 22:00:00+00:00

Columns: 82

Latest available record:


,timestamp,us_aqi,pm2_5,pm10,ozone
17470,2026-08-01 22:00:00+00:00,150,70.5,72.9,25.0


In [13]:
# ============================================================
#  VERIFY FORECASTING DATASET
# ============================================================

print("=" * 60)
print("VERIFYING FORECASTING DATASET")
print("=" * 60)

# ------------------------------------------------------------
# CHECK REQUIRED MODEL FEATURES
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in model_features
    if feature not in df_forecast.columns
]

print("Required model features:", len(model_features))
print("Missing model features:", len(missing_features))

if missing_features:
    print("\nMissing features:")
    for feature in missing_features:
        print(" -", feature)
else:
    print("All 70 model features are present")

# ------------------------------------------------------------
# TARGET CHECK
# ------------------------------------------------------------

print("\nTarget column present:", "target_aqi" in df_forecast.columns)

# ------------------------------------------------------------
# CURRENT AQI CHECK
# ------------------------------------------------------------

print("Current AQI column present:", "us_aqi" in df_forecast.columns)

# ------------------------------------------------------------
# TIMESTAMP CHECK
# ------------------------------------------------------------

print("Timestamp column present:", "timestamp" in df_forecast.columns)

# ------------------------------------------------------------
# SORT CHECK
# ------------------------------------------------------------

chronological = df_forecast["timestamp"].is_monotonic_increasing

print("Chronologically ordered:", chronological)

# ------------------------------------------------------------
# MISSING VALUES
# ------------------------------------------------------------

print("\nTotal missing values:", df_forecast.isnull().sum().sum())


VERIFYING FORECASTING DATASET
Required model features: 70
Missing model features: 0
All 70 model features are present

Target column present: True
Current AQI column present: True
Timestamp column present: True
Chronologically ordered: True

Total missing values: 0


In [14]:
# ============================================================
# DEFINE FORECAST START AND FUTURE TIMESTAMPS
# ============================================================

print("=" * 60)
print("FORECAST TIMELINE")
print("=" * 60)

# ------------------------------------------------------------
# LAST HISTORICAL OBSERVATION
# ------------------------------------------------------------

last_timestamp = df_forecast["timestamp"].iloc[-1]

last_aqi = df_forecast["us_aqi"].iloc[-1]

print("Last historical timestamp:")
print(last_timestamp)

print("\nLast observed AQI:")
print(last_aqi)

# ------------------------------------------------------------
# CREATE NEXT 72 HOURLY TIMESTAMPS
# ------------------------------------------------------------

future_timestamps = pd.date_range(
    start=last_timestamp + pd.Timedelta(hours=1),
    periods=FORECAST_HOURS,
    freq="h"
)

print("\nForecast starts:")
print(future_timestamps[0])

print("\nForecast ends:")
print(future_timestamps[-1])

print("\nNumber of forecast hours:")
print(len(future_timestamps))

# ------------------------------------------------------------
# DISPLAY FIRST AND LAST FEW
# ------------------------------------------------------------

print("\nFirst 5 future timestamps:")

for timestamp in future_timestamps[:5]:
    print(timestamp)

print("\nLast 5 future timestamps:")

for timestamp in future_timestamps[-5:]:
    print(timestamp)

FORECAST TIMELINE
Last historical timestamp:
2026-08-01 22:00:00+00:00

Last observed AQI:
150

Forecast starts:
2026-08-01 23:00:00+00:00

Forecast ends:
2026-08-04 22:00:00+00:00

Number of forecast hours:
72

First 5 future timestamps:
2026-08-01 23:00:00+00:00
2026-08-02 00:00:00+00:00
2026-08-02 01:00:00+00:00
2026-08-02 02:00:00+00:00
2026-08-02 03:00:00+00:00

Last 5 future timestamps:
2026-08-04 18:00:00+00:00
2026-08-04 19:00:00+00:00
2026-08-04 20:00:00+00:00
2026-08-04 21:00:00+00:00
2026-08-04 22:00:00+00:00


In [15]:
# ============================================================
# FETCH FUTURE OPEN-METEO FORECAST DATA
# ============================================================

import requests

print("=" * 60)
print("FETCHING FUTURE OPEN-METEO FORECAST DATA")
print("=" * 60)

LATITUDE = 34.008
LONGITUDE = 71.5785

weather_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m"
]

air_quality_variables = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# ------------------------------------------------------------
# OPEN-METEO WEATHER FORECAST
# ------------------------------------------------------------

weather_url = "https://api.open-meteo.com/v1/forecast"

weather_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": ",".join(weather_variables),
    "forecast_days": 3,
    "timezone": "UTC"
}

weather_response = requests.get(
    weather_url,
    params=weather_params,
    timeout=30
)

print("Weather API status:", weather_response.status_code)

weather_response.raise_for_status()

weather_data = weather_response.json()

# ------------------------------------------------------------
# OPEN-METEO AIR QUALITY FORECAST
# ------------------------------------------------------------

air_quality_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

air_quality_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": ",".join(air_quality_variables),
    "forecast_days": 3,
    "timezone": "UTC"
}

air_quality_response = requests.get(
    air_quality_url,
    params=air_quality_params,
    timeout=30
)

print("Air-quality API status:", air_quality_response.status_code)

air_quality_response.raise_for_status()

air_quality_data = air_quality_response.json()

print("\n✓ Open-Meteo weather data received")
print("✓ Open-Meteo air-quality data received")

FETCHING FUTURE OPEN-METEO FORECAST DATA
Weather API status: 200
Air-quality API status: 200

✓ Open-Meteo weather data received
✓ Open-Meteo air-quality data received


In [16]:
# ============================================================
# PREPARE FUTURE WEATHER + AIR QUALITY DATA
# ============================================================

print("=" * 60)
print("PREPARING FUTURE INPUT DATA")
print("=" * 60)

# ------------------------------------------------------------
# WEATHER DATAFRAME
# ------------------------------------------------------------

weather_df = pd.DataFrame(
    weather_data["hourly"]
)

weather_df["timestamp"] = pd.to_datetime(
    weather_df["time"],
    utc=True
)

weather_df = weather_df.drop(
    columns=["time"]
)

# ------------------------------------------------------------
# AIR QUALITY DATAFRAME
# ------------------------------------------------------------

air_quality_df = pd.DataFrame(
    air_quality_data["hourly"]
)

air_quality_df["timestamp"] = pd.to_datetime(
    air_quality_df["time"],
    utc=True
)

air_quality_df = air_quality_df.drop(
    columns=["time"]
)

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

future_inputs = pd.merge(
    weather_df,
    air_quality_df,
    on="timestamp",
    how="inner"
)

future_inputs = (
    future_inputs
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# KEEP ONLY OUR 72-HOUR FORECAST WINDOW
# ------------------------------------------------------------

future_inputs = future_inputs[
    future_inputs["timestamp"].isin(
        future_timestamps
    )
].copy()

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("Weather records:", len(weather_df))
print("Air-quality records:", len(air_quality_df))
print("Merged records:", len(future_inputs))

print("\nExpected forecast records:", FORECAST_HOURS)

print("\nFuture input date range:")

if len(future_inputs) > 0:
    print(
        future_inputs["timestamp"].min(),
        "→",
        future_inputs["timestamp"].max()
    )

print("\nFuture input columns:")
print(list(future_inputs.columns))

print("\nFirst 5 records:")
display(
    future_inputs.head()
)

print("\nMissing values:")
print(
    future_inputs.isnull().sum()
)

PREPARING FUTURE INPUT DATA
Weather records: 72
Air-quality records: 72
Merged records: 0

Expected forecast records: 72

Future input date range:

Future input columns:
['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'timestamp', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone']

First 5 records:


,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,timestamp,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone



Missing values:
temperature_2m          0
relative_humidity_2m    0
pressure_msl            0
precipitation           0
wind_speed_10m          0
wind_direction_10m      0
timestamp               0
pm2_5                   0
pm10                    0
carbon_monoxide         0
nitrogen_dioxide        0
sulphur_dioxide         0
ozone                   0
dtype: int64


In [17]:
# ============================================================
# FETCH HISTORICAL INPUTS FOR FORECAST BACKTEST
# ============================================================

print("=" * 60)
print("FETCHING HISTORICAL FORECAST INPUTS")
print("=" * 60)

# ------------------------------------------------------------
# BACKTEST PERIOD
# ------------------------------------------------------------

BACKTEST_START = future_timestamps[0].strftime("%Y-%m-%d")
BACKTEST_END = future_timestamps[-1].strftime("%Y-%m-%d")

print("Backtest start:", future_timestamps[0])
print("Backtest end:", future_timestamps[-1])

# ------------------------------------------------------------
# HISTORICAL WEATHER
# ------------------------------------------------------------

archive_weather_url = (
    "https://archive-api.open-meteo.com/v1/archive"
)

archive_weather_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": BACKTEST_START,
    "end_date": BACKTEST_END,
    "hourly": ",".join(weather_variables),
    "timezone": "UTC"
}

archive_weather_response = requests.get(
    archive_weather_url,
    params=archive_weather_params,
    timeout=30
)

print(
    "Historical weather API status:",
    archive_weather_response.status_code
)

archive_weather_response.raise_for_status()

archive_weather_data = archive_weather_response.json()

# ------------------------------------------------------------
# HISTORICAL AIR QUALITY
# ------------------------------------------------------------

archive_air_url = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
)

archive_air_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": BACKTEST_START,
    "end_date": BACKTEST_END,
    "hourly": ",".join(air_quality_variables),
    "timezone": "UTC"
}

archive_air_response = requests.get(
    archive_air_url,
    params=archive_air_params,
    timeout=30
)

print(
    "Historical air-quality API status:",
    archive_air_response.status_code
)

archive_air_response.raise_for_status()

archive_air_data = archive_air_response.json()

print("\n✓ Historical weather data received")
print("✓ Historical air-quality data received")

FETCHING HISTORICAL FORECAST INPUTS
Backtest start: 2026-08-01 23:00:00+00:00
Backtest end: 2026-08-04 22:00:00+00:00
Historical weather API status: 200
Historical air-quality API status: 200

✓ Historical weather data received
✓ Historical air-quality data received


In [18]:
# ============================================================
#  MERGE HISTORICAL FORECAST INPUTS
# ============================================================

print("=" * 60)
print("MERGING HISTORICAL FORECAST INPUTS")
print("=" * 60)

# ------------------------------------------------------------
# WEATHER DATAFRAME
# ------------------------------------------------------------

backtest_weather = pd.DataFrame(
    archive_weather_data["hourly"]
)

backtest_weather["timestamp"] = pd.to_datetime(
    backtest_weather["time"],
    utc=True
)

backtest_weather = backtest_weather.drop(
    columns=["time"]
)

# ------------------------------------------------------------
# AIR QUALITY DATAFRAME
# ------------------------------------------------------------

backtest_air = pd.DataFrame(
    archive_air_data["hourly"]
)

backtest_air["timestamp"] = pd.to_datetime(
    backtest_air["time"],
    utc=True
)

backtest_air = backtest_air.drop(
    columns=["time"]
)

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

backtest_inputs = pd.merge(
    backtest_weather,
    backtest_air,
    on="timestamp",
    how="inner"
)

backtest_inputs = (
    backtest_inputs
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# SELECT EXACT 72-HOUR WINDOW
# ------------------------------------------------------------

backtest_inputs = backtest_inputs[
    backtest_inputs["timestamp"].isin(
        future_timestamps
    )
].copy()

backtest_inputs = (
    backtest_inputs
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("Weather records:", len(backtest_weather))
print("Air-quality records:", len(backtest_air))
print("Merged records:", len(backtest_inputs))

print("\nExpected records:", FORECAST_HOURS)

print("\nDate range:")

if len(backtest_inputs) > 0:
    print(
        backtest_inputs["timestamp"].min(),
        "→",
        backtest_inputs["timestamp"].max()
    )

print("\nMissing values:")
print(
    backtest_inputs.isnull().sum().sum()
)

print("\nColumns:")
for i, col in enumerate(
    backtest_inputs.columns,
    start=1
):
    print(f"{i:2}. {col}")

print("\nFirst 5 records:")
display(
    backtest_inputs.head()
)

print("\nLast 5 records:")
display(
    backtest_inputs.tail()
)

MERGING HISTORICAL FORECAST INPUTS
Weather records: 96
Air-quality records: 96
Merged records: 72

Expected records: 72

Date range:
2026-08-01 23:00:00+00:00 → 2026-08-04 22:00:00+00:00

Missing values:
0

Columns:
 1. temperature_2m
 2. relative_humidity_2m
 3. pressure_msl
 4. precipitation
 5. wind_speed_10m
 6. wind_direction_10m
 7. timestamp
 8. pm2_5
 9. pm10
10. carbon_monoxide
11. nitrogen_dioxide
12. sulphur_dioxide
13. ozone

First 5 records:


,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,timestamp,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
0,25.9,93,1001.0,0.6,14.3,47,2026-08-01 23:00:00+00:00,70.1,71.9,1058.0,50.1,8.8,27.0
1,26.7,92,1001.1,3.5,4.2,317,2026-08-02 00:00:00+00:00,71.0,74.0,568.0,52.0,7.8,13.0
2,26.3,95,1002.0,0.1,6.5,275,2026-08-02 01:00:00+00:00,74.0,76.6,452.0,45.0,8.7,33.0
3,26.7,95,1002.1,0.4,4.6,347,2026-08-02 02:00:00+00:00,76.1,77.8,526.0,35.2,9.9,62.0
4,27.2,91,1002.8,2.0,7.0,35,2026-08-02 03:00:00+00:00,57.6,61.2,581.0,25.7,10.4,92.0



Last 5 records:


,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,timestamp,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
67,29.5,83,999.7,0.0,5.0,296,2026-08-04 18:00:00+00:00,79.6,83.1,649.0,56.0,11.3,42.0
68,29.0,85,999.5,0.0,4.2,290,2026-08-04 19:00:00+00:00,83.1,86.6,527.0,55.4,10.7,38.0
69,28.1,90,999.3,0.0,3.7,11,2026-08-04 20:00:00+00:00,83.1,86.6,406.0,52.0,9.4,36.0
70,27.2,93,999.0,0.1,1.8,11,2026-08-04 21:00:00+00:00,82.0,85.2,328.0,48.6,8.3,34.0
71,26.7,96,998.9,0.0,0.5,360,2026-08-04 22:00:00+00:00,81.0,84.1,316.0,46.7,7.7,29.0


In [19]:
# ============================================================
# COMBINE HISTORICAL CONTEXT + FORECAST INPUTS
# ============================================================

print("=" * 60)
print("BUILDING FORECAST FEATURE HISTORY")
print("=" * 60)

# ------------------------------------------------------------
# RAW VARIABLES REQUIRED FOR FEATURE GENERATION
# ------------------------------------------------------------

raw_columns = [
    "timestamp",
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# ------------------------------------------------------------
# GET HISTORICAL RAW DATA
# ------------------------------------------------------------

historical_raw = df_forecast[
    raw_columns
].copy()

historical_raw = (
    historical_raw
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# FUTURE RAW DATA
# ------------------------------------------------------------

future_raw = backtest_inputs[
    raw_columns
].copy()

future_raw = (
    future_raw
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

forecast_feature_history = pd.concat(
    [
        historical_raw,
        future_raw
    ],
    ignore_index=True
)

forecast_feature_history = (
    forecast_feature_history
    .sort_values("timestamp")
    .drop_duplicates(
        subset=["timestamp"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("Historical records:", len(historical_raw))
print("Future records:", len(future_raw))
print(
    "Combined records:",
    len(forecast_feature_history)
)

print("\nCombined date range:")

print(
    forecast_feature_history["timestamp"].min(),
    "→",
    forecast_feature_history["timestamp"].max()
)

print("\nDuplicate timestamps:")

print(
    forecast_feature_history["timestamp"]
    .duplicated()
    .sum()
)

print("\nMissing values:")

print(
    forecast_feature_history.isnull().sum().sum()
)

print("\nLast 5 historical records + first 5 future records:")

display(
    forecast_feature_history[
        forecast_feature_history["timestamp"]
        >= future_timestamps[0]
        - pd.Timedelta(hours=5)
    ].head(10)
)

BUILDING FORECAST FEATURE HISTORY
Historical records: 17471
Future records: 72
Combined records: 17543

Combined date range:
2024-08-04 00:00:00+00:00 → 2026-08-04 22:00:00+00:00

Duplicate timestamps:
0

Missing values:
0

Last 5 historical records + first 5 future records:


,timestamp,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
17466,2026-08-01 18:00:00+00:00,29.5,86,999.4,0.0,1.8,336,67.5,70.1,2546.0,69.0,10.7,26.0
17467,2026-08-01 19:00:00+00:00,29.2,85,999.5,0.0,3.3,315,69.6,72.2,2600.0,68.1,10.7,22.0
17468,2026-08-01 20:00:00+00:00,28.5,87,999.8,0.0,3.8,360,69.4,71.9,2508.0,63.9,10.3,23.0
17469,2026-08-01 21:00:00+00:00,27.9,90,999.9,0.0,4.5,16,69.2,71.7,2260.0,59.1,9.8,25.0
17470,2026-08-01 22:00:00+00:00,26.9,95,1000.2,0.0,6.9,6,70.5,72.9,1737.0,54.9,9.3,25.0
17471,2026-08-01 23:00:00+00:00,25.9,93,1001.0,0.6,14.3,47,70.1,71.9,1058.0,50.1,8.8,27.0
17472,2026-08-02 00:00:00+00:00,26.7,92,1001.1,3.5,4.2,317,71.0,74.0,568.0,52.0,7.8,13.0
17473,2026-08-02 01:00:00+00:00,26.3,95,1002.0,0.1,6.5,275,74.0,76.6,452.0,45.0,8.7,33.0
17474,2026-08-02 02:00:00+00:00,26.7,95,1002.1,0.4,4.6,347,76.1,77.8,526.0,35.2,9.9,62.0
17475,2026-08-02 03:00:00+00:00,27.2,91,1002.8,2.0,7.0,35,57.6,61.2,581.0,25.7,10.4,92.0


In [22]:
# ============================================================
#  PREPARE RECURSIVE FORECAST HISTORY
# ============================================================

print("=" * 60)
print("PREPARING RECURSIVE FORECAST HISTORY")
print("=" * 60)

# ------------------------------------------------------------
# Historical AQI
# ------------------------------------------------------------

historical_aqi = df_forecast[
    ["timestamp", "us_aqi"]
].copy()

historical_aqi = (
    historical_aqi
    .sort_values("timestamp")
    .drop_duplicates("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Attach historical AQI to combined raw data
# ------------------------------------------------------------

forecast_working = forecast_feature_history.copy()

forecast_working = forecast_working.merge(
    historical_aqi,
    on="timestamp",
    how="left"
)

forecast_working = (
    forecast_working
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Verify historical AQI coverage
# ------------------------------------------------------------

historical_mask = (
    forecast_working["timestamp"]
    <= last_timestamp
)

future_mask = (
    forecast_working["timestamp"]
    > last_timestamp
)

print("Historical rows:", historical_mask.sum())
print("Future rows:", future_mask.sum())

print(
    "\nHistorical AQI missing:",
    forecast_working.loc[
        historical_mask,
        "us_aqi"
    ].isna().sum()
)

print(
    "Future AQI currently missing:",
    forecast_working.loc[
        future_mask,
        "us_aqi"
    ].isna().sum()
)

print("\nLast historical AQI:")

print(
    forecast_working.loc[
        historical_mask,
        ["timestamp", "us_aqi"]
    ].tail(5)
)

print("\nFirst future rows:")

display(
    forecast_working.loc[
        future_mask,
        [
            "timestamp",
            "us_aqi",
            "pm2_5",
            "pm10",
            "ozone"
        ]
    ].head(5)
)

PREPARING RECURSIVE FORECAST HISTORY
Historical rows: 17471
Future rows: 72

Historical AQI missing: 0
Future AQI currently missing: 72

Last historical AQI:
                      timestamp  us_aqi
17466 2026-08-01 18:00:00+00:00   152.0
17467 2026-08-01 19:00:00+00:00   152.0
17468 2026-08-01 20:00:00+00:00   152.0
17469 2026-08-01 21:00:00+00:00   151.0
17470 2026-08-01 22:00:00+00:00   150.0

First future rows:


,timestamp,us_aqi,pm2_5,pm10,ozone
17471,2026-08-01 23:00:00+00:00,NaN,70.1,71.9,27.0
17472,2026-08-02 00:00:00+00:00,NaN,71.0,74.0,13.0
17473,2026-08-02 01:00:00+00:00,NaN,74.0,76.6,33.0
17474,2026-08-02 02:00:00+00:00,NaN,76.1,77.8,62.0
17475,2026-08-02 03:00:00+00:00,NaN,57.6,61.2,92.0


In [ ]:
# ============================================================
# FETCH FUTURE AIR QUALITY INCLUDING US AQI
# ============================================================

print("=" * 60)
print("FETCHING FUTURE AIR QUALITY + US AQI")
print("=" * 60)

import requests
import pandas as pd

# ------------------------------------------------------------
# Peshawar coordinates
# ------------------------------------------------------------

LATITUDE = 34.008
LONGITUDE = 71.5785

# ------------------------------------------------------------
# Forecast period
# ------------------------------------------------------------

forecast_start = (
    future_timestamps.min()
    .strftime("%Y-%m-%d")
)

forecast_end = (
    future_timestamps.max()
    .strftime("%Y-%m-%d")
)

print("Start date:", forecast_start)
print("End date:", forecast_end)

# ------------------------------------------------------------
# Request ALL pollutant variables used by the model
# PLUS us_aqi
# ------------------------------------------------------------

air_quality_url = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
)

air_quality_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,

    "hourly": ",".join([
        "pm2_5",
        "pm10",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "us_aqi"
    ]),

    "start_date": forecast_start,
    "end_date": forecast_end,

    "timezone": "GMT"
}

air_quality_response = requests.get(
    air_quality_url,
    params=air_quality_params,
    timeout=30
)

print(
    "Air-quality API status:",
    air_quality_response.status_code
)

air_quality_response.raise_for_status()

air_quality_data = air_quality_response.json()

# ------------------------------------------------------------
# Verify response
# ------------------------------------------------------------

print("\nAvailable hourly variables:")

print(
    list(
        air_quality_data["hourly"].keys()
    )
)

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

future_aqi_data = pd.DataFrame({
    "timestamp": pd.to_datetime(
        air_quality_data["hourly"]["time"],
        utc=True
    ),

    "future_us_aqi": air_quality_data[
        "hourly"
    ]["us_aqi"]
})

# ------------------------------------------------------------
# Keep exactly our required 72 forecast hours
# ------------------------------------------------------------

future_aqi_data = future_aqi_data[
    future_aqi_data["timestamp"].isin(
        future_timestamps
    )
].copy()

future_aqi_data = (
    future_aqi_data
    .sort_values("timestamp")
    .drop_duplicates("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("US AQI FORECAST VERIFICATION")
print("=" * 60)

print(
    "US AQI records:",
    len(future_aqi_data)
)

print(
    "Missing US AQI:",
    future_aqi_data["future_us_aqi"].isna().sum()
)

display(
    future_aqi_data.head(10)
)

FETCHING FUTURE AIR QUALITY + US AQI
Start date: 2026-08-01
End date: 2026-08-04
Air-quality API status: 200

Available hourly variables:
['time', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'us_aqi']

US AQI FORECAST VERIFICATION
US AQI records: 72
Missing US AQI: 0


,timestamp,future_us_aqi
0,2026-08-01 23:00:00+00:00,149
1,2026-08-02 00:00:00+00:00,147
2,2026-08-02 01:00:00+00:00,145
3,2026-08-02 02:00:00+00:00,144
4,2026-08-02 03:00:00+00:00,142
5,2026-08-02 04:00:00+00:00,142
6,2026-08-02 05:00:00+00:00,141
7,2026-08-02 06:00:00+00:00,140
8,2026-08-02 07:00:00+00:00,139
9,2026-08-02 08:00:00+00:00,139


In [36]:
# ============================================================
# MERGE FUTURE US AQI INTO FORECAST HISTORY
# ============================================================

print("=" * 60)
print("MERGING FUTURE US AQI")
print("=" * 60)

# Remove it first in case this cell is accidentally rerun
forecast_working = forecast_working.drop(
    columns=["future_us_aqi"],
    errors="ignore"
)

forecast_working = forecast_working.merge(
    future_aqi_data[
        [
            "timestamp",
            "future_us_aqi"
        ]
    ],
    on="timestamp",
    how="left"
)

forecast_working = (
    forecast_working
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

future_mask = (
    forecast_working["timestamp"]
    > last_timestamp
)

print("\nCombined rows:", len(forecast_working))

print(
    "Future rows:",
    future_mask.sum()
)

print(
    "Future US AQI missing:",
    forecast_working.loc[
        future_mask,
        "future_us_aqi"
    ].isna().sum()
)

print("\nLast 5 historical + first 5 future:")

display(
    forecast_working.loc[
        (
            forecast_working["timestamp"]
            >= last_timestamp - pd.Timedelta(hours=4)
        ),
        [
            "timestamp",
            "us_aqi",
            "future_us_aqi",
            "pm2_5",
            "pm10",
            "ozone"
        ]
    ].head(9)
)



assert len(forecast_working) == 17543
assert future_mask.sum() == 72
assert (
    forecast_working.loc[
        future_mask,
        "future_us_aqi"
    ].isna().sum() == 0
)


MERGING FUTURE US AQI

Combined rows: 17543
Future rows: 72
Future US AQI missing: 0

Last 5 historical + first 5 future:


,timestamp,us_aqi,future_us_aqi,pm2_5,pm10,ozone
17466,2026-08-01 18:00:00+00:00,152.0,NaN,67.5,70.1,26.0
17467,2026-08-01 19:00:00+00:00,152.0,NaN,69.6,72.2,22.0
17468,2026-08-01 20:00:00+00:00,152.0,NaN,69.4,71.9,23.0
17469,2026-08-01 21:00:00+00:00,151.0,NaN,69.2,71.7,25.0
17470,2026-08-01 22:00:00+00:00,150.0,NaN,70.5,72.9,25.0
17471,2026-08-01 23:00:00+00:00,NaN,149.0,70.1,71.9,27.0
17472,2026-08-02 00:00:00+00:00,NaN,147.0,71.0,74.0,13.0
17473,2026-08-02 01:00:00+00:00,NaN,145.0,74.0,76.6,33.0
17474,2026-08-02 02:00:00+00:00,NaN,144.0,76.1,77.8,62.0


In [44]:
# ============================================================
#  PREPARE RECURSIVE FORECASTING
# ============================================================

print("=" * 60)
print("PREPARING RECURSIVE XGBOOST FORECAST")
print("=" * 60)

# ------------------------------------------------------------
# Get exact champion feature list
# ------------------------------------------------------------

if hasattr(champion_model, "feature_names_in_"):
    feature_columns = list(
        champion_model.feature_names_in_
    )
else:
    feature_columns = list(
        champion_model.get_booster().feature_names
    )

print("Champion features:", len(feature_columns))

assert len(feature_columns) == 70


# ------------------------------------------------------------
# Historical + future working data
# ------------------------------------------------------------

forecast_working = (
    forecast_working
    .sort_values("timestamp")
    .reset_index(drop=True)
)

future_mask = (
    forecast_working["timestamp"]
    > last_timestamp
)

future_positions = (
    forecast_working.index[future_mask]
    .tolist()
)

print("Historical rows:", (~future_mask).sum())
print("Future rows:", len(future_positions))


# ------------------------------------------------------------
# Recursive AQI history
# ------------------------------------------------------------

recursive_aqi = (
    forecast_working["us_aqi"]
    .copy()
)

# Future AQI is unknown initially.
recursive_aqi.loc[future_mask] = np.nan


# ------------------------------------------------------------
# Verify historical AQI
# ------------------------------------------------------------

print(
    "\nHistorical AQI missing:",
    recursive_aqi.iloc[:future_positions[0]].isna().sum()
)

print(
    "Future AQI initially missing:",
    recursive_aqi.iloc[future_positions].isna().sum()
)

print(
    "\nLast observed AQI:",
    recursive_aqi.iloc[
        future_positions[0] - 1
    ]
)



PREPARING RECURSIVE XGBOOST FORECAST
Champion features: 70
Historical rows: 17471
Future rows: 72

Historical AQI missing: 0
Future AQI initially missing: 72

Last observed AQI: 150.0


In [46]:
# ============================================================
# RECURSIVE 72-HOUR XGBOOST FORECAST
# ============================================================

print("=" * 60)
print("RECURSIVE 72-HOUR XGBOOST FORECAST")
print("=" * 60)

# ------------------------------------------------------------
# STORAGE
# ------------------------------------------------------------

forecast_predictions = []
forecast_feature_rows = []

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------

def get_recursive_aqi(position, lag):

    previous_position = position - lag

    if previous_position < 0:
        return np.nan

    return recursive_aqi.iloc[previous_position]


# ============================================================
# FORECAST HOUR BY HOUR
# ============================================================

for forecast_number, position in enumerate(
    future_positions,
    start=1
):

    row = forecast_working.iloc[position]

    timestamp = pd.Timestamp(
        row["timestamp"]
    )

    # ========================================================
    # BUILD EXACTLY THE SAME 70 FEATURES
    # ========================================================

    features = {}

    # --------------------------------------------------------
    # WEATHER
    # --------------------------------------------------------

    features["temperature_2m"] = row["temperature_2m"]
    features["relative_humidity_2m"] = row["relative_humidity_2m"]
    features["pressure_msl"] = row["pressure_msl"]
    features["precipitation"] = row["precipitation"]
    features["wind_speed_10m"] = row["wind_speed_10m"]
    features["wind_direction_10m"] = row["wind_direction_10m"]

    # --------------------------------------------------------
    # POLLUTANTS
    # --------------------------------------------------------

    pollutant_names = [
        "pm2_5",
        "pm10",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone"
    ]

    for pollutant in pollutant_names:
        features[pollutant] = row[pollutant]

    # --------------------------------------------------------
    # TIME FEATURES
    # --------------------------------------------------------

    features["hour"] = timestamp.hour
    features["day_of_week"] = timestamp.dayofweek
    features["day_of_month"] = timestamp.day
    features["month"] = timestamp.month
    features["is_weekend"] = int(
        timestamp.dayofweek >= 5
    )

    # --------------------------------------------------------
    # AQI LAGS
    # --------------------------------------------------------

    for lag in [1, 3, 6, 12, 24, 48, 72]:

        features[
            f"aqi_lag_{lag}"
        ] = get_recursive_aqi(
            position,
            lag
        )

    # --------------------------------------------------------
    # POLLUTANT LAGS
    # --------------------------------------------------------

    for pollutant in pollutant_names:

        for lag in [1, 3, 6, 24]:

            previous_position = position - lag

            features[
                f"{pollutant}_lag_{lag}"
            ] = forecast_working.iloc[
                previous_position
            ][pollutant]

    # --------------------------------------------------------
    # AQI ROLLING MEANS
    # --------------------------------------------------------

    for window in [3, 6, 12, 24]:

        start = position - window + 1

        aqi_values = recursive_aqi.iloc[
            start:position + 1
        ]

        features[
            f"aqi_{window}h_mean"
        ] = aqi_values.mean()

    # --------------------------------------------------------
    # PM2.5 ROLLING MEANS
    # --------------------------------------------------------

    for window in [3, 6, 24]:

        start = position - window + 1

        values = forecast_working.iloc[
            start:position + 1
        ]["pm2_5"]

        features[
            f"pm2_5_{window}h_mean"
        ] = values.mean()

    # --------------------------------------------------------
    # PM10 ROLLING MEANS
    # --------------------------------------------------------

    for window in [3, 6, 24]:

        start = position - window + 1

        values = forecast_working.iloc[
            start:position + 1
        ]["pm10"]

        features[
            f"pm10_{window}h_mean"
        ] = values.mean()

    # --------------------------------------------------------
    # OTHER 24-HOUR POLLUTANT MEANS
    # --------------------------------------------------------

    for pollutant in [
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone"
    ]:

        start = position - 24 + 1

        values = forecast_working.iloc[
            start:position + 1
        ][pollutant]

        features[
            f"{pollutant}_24h_mean"
        ] = values.mean()

    # --------------------------------------------------------
    # AQI CHANGE FEATURES
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # These must be based on the recursive AQI history,
    # NOT future_us_aqi.
    #
    # The current AQI for this prediction is represented by
    # the latest observed/predicted AQI at position - 1.
    # --------------------------------------------------------

    current_aqi = get_recursive_aqi(
        position,
        1
    )

    for lag in [1, 3, 6, 24]:

        previous_aqi = get_recursive_aqi(
            position,
            lag + 1
        )

        features[
            f"aqi_change_{lag}h"
        ] = (
            current_aqi
            - previous_aqi
        )

    # --------------------------------------------------------
    # PM2.5 CHANGES
    # --------------------------------------------------------

    current_pm25 = row["pm2_5"]

    previous_pm25_1 = forecast_working.iloc[
        position - 1
    ]["pm2_5"]

    previous_pm25_24 = forecast_working.iloc[
        position - 24
    ]["pm2_5"]

    features["pm2_5_change_1h"] = (
        current_pm25
        - previous_pm25_1
    )

    features["pm2_5_change_24h"] = (
        current_pm25
        - previous_pm25_24
    )

    # --------------------------------------------------------
    # PM10 CHANGES
    # --------------------------------------------------------

    current_pm10 = row["pm10"]

    previous_pm10_1 = forecast_working.iloc[
        position - 1
    ]["pm10"]

    previous_pm10_24 = forecast_working.iloc[
        position - 24
    ]["pm10"]

    features["pm10_change_1h"] = (
        current_pm10
        - previous_pm10_1
    )

    features["pm10_change_24h"] = (
        current_pm10
        - previous_pm10_24
    )

    # ========================================================
    # CREATE SINGLE FEATURE ROW
    # ========================================================

    X_future_row = pd.DataFrame(
        [features]
    )

    # EXACT CHAMPION ORDER
    X_future_row = X_future_row[
        feature_columns
    ]

    # --------------------------------------------------------
    # SANITY CHECK BEFORE PREDICTION
    # --------------------------------------------------------

    if X_future_row.isna().sum().sum() != 0:

        missing_features = (
            X_future_row.columns[
                X_future_row.isna().any()
            ].tolist()
        )

        raise ValueError(
            f"NaN detected at forecast hour "
            f"{forecast_number} "
            f"({timestamp}). "
            f"Missing features: "
            f"{missing_features}"
        )

    # ========================================================
    # XGBOOST PREDICTION
    # ========================================================

    prediction = float(
        champion_model.predict(
            X_future_row
        )[0]
    )

    # --------------------------------------------------------
    # AQI BOUNDARY
    # --------------------------------------------------------

    prediction = max(
        0.0,
        prediction
    )

    # --------------------------------------------------------
    # STORE PREDICTION
    # --------------------------------------------------------

    recursive_aqi.iloc[
        position
    ] = prediction

    forecast_predictions.append(
        prediction
    )

    forecast_feature_rows.append(
        X_future_row.iloc[0].copy()
    )

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if (
        forecast_number <= 5
        or forecast_number % 12 == 0
        or forecast_number == 72
    ):

        print(
            f"{forecast_number:02d}/72 | "
            f"{timestamp} | "
            f"Predicted AQI: "
            f"{prediction:.2f}"
        )


# ============================================================
# CREATE FINAL FORECAST DATAFRAME
# ============================================================

forecast_features_final = pd.DataFrame(
    forecast_feature_rows
)

forecast_features_final = (
    forecast_features_final[
        feature_columns
    ]
    .reset_index(drop=True)
)

forecast_predictions = np.array(
    forecast_predictions
)

forecast_timestamps = (
    forecast_working.loc[
        future_positions,
        "timestamp"
    ]
    .reset_index(drop=True)
)

forecast_results = pd.DataFrame({
    "timestamp": forecast_timestamps,
    "predicted_aqi": forecast_predictions
})


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("RECURSIVE FORECAST VERIFICATION")
print("=" * 60)

print(
    "Forecast rows:",
    len(forecast_results)
)

print(
    "Forecast feature shape:",
    forecast_features_final.shape
)

print(
    "Missing forecast features:",
    forecast_features_final.isna().sum().sum()
)

print(
    "Missing predictions:",
    pd.isna(
        forecast_results["predicted_aqi"]
    ).sum()
)

print(
    "Timestamp order:",
    forecast_results[
        "timestamp"
    ].is_monotonic_increasing
)

assert len(forecast_results) == 72

assert forecast_features_final.shape == (
    72,
    70
)

assert (
    forecast_features_final.isna().sum().sum()
    == 0
)

assert (
    forecast_results["predicted_aqi"].isna().sum()
    == 0
)

assert (
    forecast_results[
        "timestamp"
    ].is_monotonic_increasing
)


print("\nFirst 10 forecasts:")

display(
    forecast_results.head(10)
)

print("\nLast 10 forecasts:")

display(
    forecast_results.tail(10)
)

RECURSIVE 72-HOUR XGBOOST FORECAST
01/72 | 2026-08-01 23:00:00+00:00 | Predicted AQI: 146.18
02/72 | 2026-08-02 00:00:00+00:00 | Predicted AQI: 141.86
03/72 | 2026-08-02 01:00:00+00:00 | Predicted AQI: 137.92
04/72 | 2026-08-02 02:00:00+00:00 | Predicted AQI: 136.75
05/72 | 2026-08-02 03:00:00+00:00 | Predicted AQI: 138.69
12/72 | 2026-08-02 10:00:00+00:00 | Predicted AQI: 144.32
24/72 | 2026-08-02 22:00:00+00:00 | Predicted AQI: 115.65
36/72 | 2026-08-03 10:00:00+00:00 | Predicted AQI: 161.96
48/72 | 2026-08-03 22:00:00+00:00 | Predicted AQI: 146.82
60/72 | 2026-08-04 10:00:00+00:00 | Predicted AQI: 152.11
72/72 | 2026-08-04 22:00:00+00:00 | Predicted AQI: 155.54

RECURSIVE FORECAST VERIFICATION
Forecast rows: 72
Forecast feature shape: (72, 70)
Missing forecast features: 0
Missing predictions: 0
Timestamp order: True

First 10 forecasts:


,timestamp,predicted_aqi
0,2026-08-01 23:00:00+00:00,146.176331
1,2026-08-02 00:00:00+00:00,141.855682
2,2026-08-02 01:00:00+00:00,137.918091
3,2026-08-02 02:00:00+00:00,136.754822
4,2026-08-02 03:00:00+00:00,138.686417
5,2026-08-02 04:00:00+00:00,140.537521
6,2026-08-02 05:00:00+00:00,140.265747
7,2026-08-02 06:00:00+00:00,138.260590
8,2026-08-02 07:00:00+00:00,136.773529
9,2026-08-02 08:00:00+00:00,136.985748



Last 10 forecasts:


,timestamp,predicted_aqi
62,2026-08-04 13:00:00+00:00,154.820435
63,2026-08-04 14:00:00+00:00,150.043365
64,2026-08-04 15:00:00+00:00,147.799850
65,2026-08-04 16:00:00+00:00,147.853836
66,2026-08-04 17:00:00+00:00,149.219421
67,2026-08-04 18:00:00+00:00,150.385391
68,2026-08-04 19:00:00+00:00,151.743866
69,2026-08-04 20:00:00+00:00,152.540512
70,2026-08-04 21:00:00+00:00,153.048019
71,2026-08-04 22:00:00+00:00,155.543213


In [50]:
# ============================================================
# THREE 24-HOUR AQI FORECAST DAYS
# ============================================================

print("=" * 60)
print("THREE-DAY AQI FORECAST SUMMARY")
print("=" * 60)

# ------------------------------------------------------------
# IMPORTANT:
# Define forecast days as 3 consecutive 24-hour periods
# beginning at the actual forecast start timestamp.
# ------------------------------------------------------------

forecast_results = forecast_results.sort_values(
    "timestamp"
).reset_index(drop=True)


assert len(forecast_results) == 72


# ------------------------------------------------------------
# Assign Day 1 / Day 2 / Day 3
# ------------------------------------------------------------

forecast_results["forecast_day"] = (
    np.arange(len(forecast_results)) // 24
) + 1


# ------------------------------------------------------------
# AQI CATEGORY
# ------------------------------------------------------------

def aqi_category(aqi):

    if aqi <= 50:
        return "Good"

    elif aqi <= 100:
        return "Moderate"

    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"

    elif aqi <= 200:
        return "Unhealthy"

    elif aqi <= 300:
        return "Very Unhealthy"

    else:
        return "Hazardous"


# ------------------------------------------------------------
# DAILY SUMMARY
# ------------------------------------------------------------

daily_forecast = (
    forecast_results
    .groupby("forecast_day")
    .agg(
        forecast_start=(
            "timestamp",
            "min"
        ),
        forecast_end=(
            "timestamp",
            "max"
        ),
        average_aqi=(
            "predicted_aqi",
            "mean"
        ),
        minimum_aqi=(
            "predicted_aqi",
            "min"
        ),
        maximum_aqi=(
            "predicted_aqi",
            "max"
        ),
        hours=(
            "predicted_aqi",
            "count"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# AQI CATEGORY
# ------------------------------------------------------------

daily_forecast["category"] = (
    daily_forecast["average_aqi"]
    .apply(aqi_category)
)


# ------------------------------------------------------------
# ROUND VALUES FOR DISPLAY
# ------------------------------------------------------------

daily_forecast[
    [
        "average_aqi",
        "minimum_aqi",
        "maximum_aqi"
    ]
] = daily_forecast[
    [
        "average_aqi",
        "minimum_aqi",
        "maximum_aqi"
    ]
].round(2)


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 60)
print("DAILY AQI FORECAST")
print("=" * 60)

display(
    daily_forecast
)


# ============================================================
# VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("DAILY FORECAST VERIFICATION")
print("=" * 60)

print(
    "Forecast days:",
    len(daily_forecast)
)

print(
    "Total forecast hours:",
    daily_forecast["hours"].sum()
)

print(
    "Hours per forecast day:",
    daily_forecast["hours"].tolist()
)

assert len(daily_forecast) == 3

assert (
    daily_forecast["hours"].tolist()
    == [24, 24, 24]
)

assert (
    daily_forecast["hours"].sum()
    == 72
)


THREE-DAY AQI FORECAST SUMMARY

DAILY AQI FORECAST


,forecast_day,forecast_start,forecast_end,average_aqi,minimum_aqi,maximum_aqi,hours,category
0,1,2026-08-01 23:00:00+00:00,2026-08-02 22:00:00+00:00,135.92,115.65,156.09,24,Unhealthy for Sensitive Groups
1,2,2026-08-02 23:00:00+00:00,2026-08-03 22:00:00+00:00,131.99,106.61,172.32,24,Unhealthy for Sensitive Groups
2,3,2026-08-03 23:00:00+00:00,2026-08-04 22:00:00+00:00,151.36,145.02,156.06,24,Unhealthy



DAILY FORECAST VERIFICATION
Forecast days: 3
Total forecast hours: 72
Hours per forecast day: [24, 24, 24]


In [52]:
# ============================================================
# ATTACH ACTUAL AQI AND EVALUATE 72-HOUR FORECAST
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd
import os

print("=" * 60)
print("ATTACHING ACTUAL AQI TO FORECAST")
print("=" * 60)


# ============================================================
# 1. CHECK FORECAST RESULTS
# ============================================================

print("Forecast rows:", len(forecast_results))

print("\nForecast columns:")
print(forecast_results.columns.tolist())

assert "timestamp" in forecast_results.columns
assert "predicted_aqi" in forecast_results.columns


# ============================================================
# 2. FIND ACTUAL FUTURE AQI DATA
# ============================================================

print("\nSearching for actual future AQI...")

# The variable created earlier in your pipeline should contain
# the 72-hour Open-Meteo US AQI values.

possible_actuals = [
    "future_aqi_data",
    "future_us_aqi_data",
    "future_air_quality",
    "air_quality_data"
]

actual_source = None

for variable_name in possible_actuals:

    if variable_name in globals():

        obj = globals()[variable_name]

        if isinstance(obj, pd.DataFrame):

            print(
                f"Found DataFrame: {variable_name}"
            )

            print(
                "Columns:",
                obj.columns.tolist()
            )

            if "timestamp" in obj.columns:

                if "future_us_aqi" in obj.columns:
                    actual_source = obj[
                        [
                            "timestamp",
                            "future_us_aqi"
                        ]
                    ].copy()
                    break

                elif "us_aqi" in obj.columns:
                    actual_source = obj[
                        [
                            "timestamp",
                            "us_aqi"
                        ]
                    ].copy()

                    actual_source = (
                        actual_source
                        .rename(
                            columns={
                                "us_aqi":
                                "future_us_aqi"
                            }
                        )
                    )

                    break


# ============================================================
# 3. IF NOT FOUND, CHECK COMMON DATAFRAME NAMES
# ============================================================

if actual_source is None:

    print(
        "\nCould not automatically identify "
        "the actual AQI DataFrame."
    )

    print(
        "\nAvailable DataFrames containing "
        "'aqi' or 'air':"
    )

    for name, obj in globals().items():

        if isinstance(obj, pd.DataFrame):

            name_lower = name.lower()

            if (
                "aqi" in name_lower
                or "air" in name_lower
            ):

                print(
                    name,
                    obj.shape,
                    obj.columns.tolist()
                )

    raise ValueError(
        "Actual future AQI DataFrame not found. "
        "Send me the output above."
    )


# ============================================================
# 4. NORMALIZE TIMESTAMPS
# ============================================================

actual_source["timestamp"] = pd.to_datetime(
    actual_source["timestamp"],
    utc=True
)

forecast_results["timestamp"] = pd.to_datetime(
    forecast_results["timestamp"],
    utc=True
)


# ============================================================
# 5. KEEP ONLY FORECAST PERIOD
# ============================================================

actual_source = actual_source[
    actual_source["timestamp"].isin(
        forecast_results["timestamp"]
    )
].copy()


print(
    "\nActual AQI rows matched:",
    len(actual_source)
)

assert len(actual_source) == 72


# ============================================================
# 6. MERGE ACTUAL AQI WITH PREDICTIONS
# ============================================================

evaluation_df = forecast_results[
    [
        "timestamp",
        "predicted_aqi"
    ]
].copy()

evaluation_df = evaluation_df.merge(
    actual_source,
    on="timestamp",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 7. VERIFY MERGE
# ============================================================

print("\n" + "=" * 60)
print("EVALUATION DATA VERIFICATION")
print("=" * 60)

print(
    "Evaluation rows:",
    len(evaluation_df)
)

print(
    "Missing actual AQI:",
    evaluation_df["future_us_aqi"].isna().sum()
)

print(
    "Missing predictions:",
    evaluation_df["predicted_aqi"].isna().sum()
)

assert len(evaluation_df) == 72

assert (
    evaluation_df["future_us_aqi"]
    .isna()
    .sum()
    == 0
)

assert (
    evaluation_df["predicted_aqi"]
    .isna()
    .sum()
    == 0
)


# ============================================================
# 8. CALCULATE METRICS
# ============================================================

y_true = evaluation_df[
    "future_us_aqi"
].values

y_pred = evaluation_df[
    "predicted_aqi"
].values


mae = mean_absolute_error(
    y_true,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)

r2 = r2_score(
    y_true,
    y_pred
)


# ============================================================
# 9. RESULTS
# ============================================================

print("\n" + "=" * 60)
print("72-HOUR FORECAST PERFORMANCE")
print("=" * 60)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


# ============================================================
# 10. SHOW FIRST 10 COMPARISONS
# ============================================================

print("\nFirst 10 predictions vs actual:")

display(
    evaluation_df.head(10)
)


# ============================================================
# 11. SAVE
# ============================================================

os.makedirs(
    "models/evaluation",
    exist_ok=True
)

evaluation_df.to_csv(
    "models/evaluation/xgboost_72h_forecast.csv",
    index=False
)

print("\nSaved:")
print(
    "models/evaluation/xgboost_72h_forecast.csv"
)

ATTACHING ACTUAL AQI TO FORECAST
Forecast rows: 72

Forecast columns:
['timestamp', 'predicted_aqi', 'date', 'forecast_day']

Searching for actual future AQI...
Found DataFrame: future_aqi_data
Columns: ['timestamp', 'future_us_aqi']

Actual AQI rows matched: 72

EVALUATION DATA VERIFICATION
Evaluation rows: 72
Missing actual AQI: 0
Missing predictions: 0

72-HOUR FORECAST PERFORMANCE
MAE  : 4.4579
RMSE : 6.9574
R²   : 0.7790

First 10 predictions vs actual:


,timestamp,predicted_aqi,future_us_aqi
0,2026-08-01 23:00:00+00:00,146.176331,149
1,2026-08-02 00:00:00+00:00,141.855682,147
2,2026-08-02 01:00:00+00:00,137.918091,145
3,2026-08-02 02:00:00+00:00,136.754822,144
4,2026-08-02 03:00:00+00:00,138.686417,142
5,2026-08-02 04:00:00+00:00,140.537521,142
6,2026-08-02 05:00:00+00:00,140.265747,141
7,2026-08-02 06:00:00+00:00,138.260590,140
8,2026-08-02 07:00:00+00:00,136.773529,139
9,2026-08-02 08:00:00+00:00,136.985748,139



Saved:
models/evaluation/xgboost_72h_forecast.csv


In [54]:
# ============================================================
#  DAILY AQI FORECAST EVALUATION
# ============================================================

print("=" * 60)
print("DAILY AQI FORECAST EVALUATION")
print("=" * 60)


# ============================================================
# 1. PREPARE EVALUATION DATA
# ============================================================

daily_eval = evaluation_df.copy()

daily_eval = daily_eval.sort_values(
    "timestamp"
).reset_index(drop=True)


# Make sure we have exactly 72 hourly records
assert len(daily_eval) == 72


# ============================================================
# 2. ASSIGN THE SAME 3 FORECAST DAYS
# ============================================================

daily_eval["forecast_day"] = (
    np.arange(len(daily_eval)) // 24
) + 1


# ============================================================
# 3. CALCULATE DAILY PREDICTED + ACTUAL AVERAGES
# ============================================================

daily_evaluation = (
    daily_eval
    .groupby("forecast_day")
    .agg(
        forecast_start=(
            "timestamp",
            "min"
        ),

        forecast_end=(
            "timestamp",
            "max"
        ),

        predicted_average_aqi=(
            "predicted_aqi",
            "mean"
        ),

        actual_average_aqi=(
            "future_us_aqi",
            "mean"
        ),

        predicted_min_aqi=(
            "predicted_aqi",
            "min"
        ),

        actual_min_aqi=(
            "future_us_aqi",
            "min"
        ),

        predicted_max_aqi=(
            "predicted_aqi",
            "max"
        ),

        actual_max_aqi=(
            "future_us_aqi",
            "max"
        ),

        hours=(
            "predicted_aqi",
            "count"
        )
    )
    .reset_index()
)


# ============================================================
# 4. DAILY ERROR
# ============================================================

daily_evaluation["absolute_error"] = (
    daily_evaluation["predicted_average_aqi"]
    -
    daily_evaluation["actual_average_aqi"]
).abs()


daily_evaluation["percentage_error"] = (
    daily_evaluation["absolute_error"]
    /
    daily_evaluation["actual_average_aqi"]
    * 100
)


# ============================================================
# 5. ROUND VALUES
# ============================================================

numeric_columns = [
    "predicted_average_aqi",
    "actual_average_aqi",
    "predicted_min_aqi",
    "actual_min_aqi",
    "predicted_max_aqi",
    "actual_max_aqi",
    "absolute_error",
    "percentage_error"
]

daily_evaluation[numeric_columns] = (
    daily_evaluation[numeric_columns]
    .round(2)
)


# ============================================================
# 6. DISPLAY DAILY RESULTS
# ============================================================

print("\n" + "=" * 60)
print("DAILY PREDICTED VS ACTUAL AQI")
print("=" * 60)

display(
    daily_evaluation
)


# ============================================================
# 7. DAILY MAE
# ============================================================

daily_mae = mean_absolute_error(
    daily_evaluation["actual_average_aqi"],
    daily_evaluation["predicted_average_aqi"]
)


daily_rmse = np.sqrt(
    mean_squared_error(
        daily_evaluation["actual_average_aqi"],
        daily_evaluation["predicted_average_aqi"]
    )
)


daily_r2 = r2_score(
    daily_evaluation["actual_average_aqi"],
    daily_evaluation["predicted_average_aqi"]
)


# ============================================================
# 8. PRINT DAILY METRICS
# ============================================================

print("\n" + "=" * 60)
print("DAILY AVERAGE AQI PERFORMANCE")
print("=" * 60)

print(
    f"Daily MAE  : {daily_mae:.4f}"
)

print(
    f"Daily RMSE : {daily_rmse:.4f}"
)

print(
    f"Daily R²   : {daily_r2:.4f}"
)


# ============================================================
# 9. VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("DAILY EVALUATION VERIFICATION")
print("=" * 60)

print(
    "Forecast days:",
    len(daily_evaluation)
)

print(
    "Hours per day:",
    daily_evaluation["hours"].tolist()
)

assert len(daily_evaluation) == 3

assert (
    daily_evaluation["hours"].tolist()
    == [24, 24, 24]
)

assert (
    daily_evaluation["predicted_average_aqi"]
    .notna()
    .all()
)

assert (
    daily_evaluation["actual_average_aqi"]
    .notna()
    .all()
)



DAILY AQI FORECAST EVALUATION

DAILY PREDICTED VS ACTUAL AQI


,forecast_day,forecast_start,forecast_end,predicted_average_aqi,actual_average_aqi,predicted_min_aqi,actual_min_aqi,predicted_max_aqi,actual_max_aqi,hours,absolute_error,percentage_error
0,1,2026-08-01 23:00:00+00:00,2026-08-02 22:00:00+00:00,135.92,138.92,115.65,122,156.09,157,24,3.00,2.16
1,2,2026-08-02 23:00:00+00:00,2026-08-03 22:00:00+00:00,131.99,129.88,106.61,108,172.32,165,24,2.11,1.63
2,3,2026-08-03 23:00:00+00:00,2026-08-04 22:00:00+00:00,151.36,151.58,145.02,148,156.06,153,24,0.23,0.15



DAILY AVERAGE AQI PERFORMANCE
Daily MAE  : 1.7767
Daily RMSE : 2.1214
Daily R²   : 0.9432

DAILY EVALUATION VERIFICATION
Forecast days: 3
Hours per day: [24, 24, 24]


In [56]:
# ============================================================
#  SEPARATE EVALUATION FOR DAY 1, DAY 2, DAY 3
# ============================================================

print("=" * 60)
print("SEPARATE 24-HOUR EVALUATION")
print("=" * 60)

# ------------------------------------------------------------
# VERIFY DATA
# ------------------------------------------------------------

assert len(evaluation_df) == 72

day_eval = evaluation_df.copy()

day_eval = day_eval.sort_values(
    "timestamp"
).reset_index(drop=True)

# Exactly 24 hours per forecast day
day_eval["forecast_day"] = (
    np.arange(len(day_eval)) // 24
) + 1

assert (
    day_eval["forecast_day"]
    .value_counts()
    .sort_index()
    .tolist()
    == [24, 24, 24]
)


# ============================================================
# CALCULATE METRICS FOR EACH DAY
# ============================================================

daily_metrics = []

for day in [1, 2, 3]:

    day_data = day_eval[
        day_eval["forecast_day"] == day
    ].copy()

    y_true_day = day_data[
        "future_us_aqi"
    ].values

    y_pred_day = day_data[
        "predicted_aqi"
    ].values

    # ----------------------------------------
    # HOURLY PREDICTION METRICS FOR THIS DAY
    # ----------------------------------------

    day_mae = mean_absolute_error(
        y_true_day,
        y_pred_day
    )

    day_rmse = np.sqrt(
        mean_squared_error(
            y_true_day,
            y_pred_day
        )
    )

    day_r2 = r2_score(
        y_true_day,
        y_pred_day
    )

    # ----------------------------------------
    # DAILY AVERAGES
    # ----------------------------------------

    predicted_avg = np.mean(
        y_pred_day
    )

    actual_avg = np.mean(
        y_true_day
    )

    average_error = (
        predicted_avg
        - actual_avg
    )

    absolute_average_error = abs(
        average_error
    )

    percentage_error = (
        absolute_average_error
        / actual_avg
        * 100
    )

    daily_metrics.append({

        "Forecast Day": f"Day {day}",

        "Start": day_data[
            "timestamp"
        ].min(),

        "End": day_data[
            "timestamp"
        ].max(),

        "Hours": len(day_data),

        "Predicted Avg AQI":
            predicted_avg,

        "Actual Avg AQI":
            actual_avg,

        "Average Error":
            average_error,

        "Absolute Avg Error":
            absolute_average_error,

        "Percentage Error (%)":
            percentage_error,

        "MAE":
            day_mae,

        "RMSE":
            day_rmse,

        "R²":
            day_r2
    })


# ============================================================
# CREATE RESULTS TABLE
# ============================================================

separate_day_evaluation = pd.DataFrame(
    daily_metrics
)


# ============================================================
# ROUND FOR DISPLAY
# ============================================================

display_columns = [
    "Predicted Avg AQI",
    "Actual Avg AQI",
    "Average Error",
    "Absolute Avg Error",
    "Percentage Error (%)",
    "MAE",
    "RMSE",
    "R²"
]

separate_day_evaluation[
    display_columns
] = separate_day_evaluation[
    display_columns
].round(4)


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 60)
print("DAY-BY-DAY FORECAST EVALUATION")
print("=" * 60)

display(
    separate_day_evaluation
)


# ============================================================
# PRINT EACH DAY INDIVIDUALLY
# ============================================================

for _, row in separate_day_evaluation.iterrows():

    print("\n" + "-" * 60)
    print(row["Forecast Day"])
    print("-" * 60)

    print(
        "Start:",
        row["Start"]
    )

    print(
        "End:",
        row["End"]
    )

    print(
        "Hours:",
        row["Hours"]
    )

    print(
        f"Predicted Average AQI : "
        f"{row['Predicted Avg AQI']:.4f}"
    )

    print(
        f"Actual Average AQI    : "
        f"{row['Actual Avg AQI']:.4f}"
    )

    print(
        f"Average Error         : "
        f"{row['Average Error']:.4f}"
    )

    print(
        f"Absolute Avg Error    : "
        f"{row['Absolute Avg Error']:.4f}"
    )

    print(
        f"Percentage Error      : "
        f"{row['Percentage Error (%)']:.2f}%"
    )

    print(
        f"Hourly MAE            : "
        f"{row['MAE']:.4f}"
    )

    print(
        f"Hourly RMSE           : "
        f"{row['RMSE']:.4f}"
    )

    print(
        f"Hourly R²             : "
        f"{row['R²']:.4f}"
    )


# ============================================================
# SAVE
# ============================================================

os.makedirs(
    "models/evaluation",
    exist_ok=True
)

separate_day_evaluation.to_csv(
    "models/evaluation/xgboost_daily_evaluation.csv",
    index=False
)

print("\n" + "=" * 60)
print("SAVED")
print("=" * 60)

print(
    "models/evaluation/xgboost_daily_evaluation.csv"
)

SEPARATE 24-HOUR EVALUATION

DAY-BY-DAY FORECAST EVALUATION


,Forecast Day,Start,End,Hours,Predicted Avg AQI,Actual Avg AQI,Average Error,Absolute Avg Error,Percentage Error (%),MAE,RMSE,R²
0,Day 1,2026-08-01 23:00:00+00:00,2026-08-02 22:00:00+00:00,24,135.9210,138.9167,-2.9957,2.9957,2.1565,4.9470,5.9467,0.5868
1,Day 2,2026-08-02 23:00:00+00:00,2026-08-03 22:00:00+00:00,24,131.9865,129.8750,2.1115,2.1115,1.6258,6.9970,10.2875,0.6813
2,Day 3,2026-08-03 23:00:00+00:00,2026-08-04 22:00:00+00:00,24,151.3583,151.5833,-0.2251,0.2251,0.1485,1.4298,2.0049,-1.4219



------------------------------------------------------------
Day 1
------------------------------------------------------------
Start: 2026-08-01 23:00:00+00:00
End: 2026-08-02 22:00:00+00:00
Hours: 24
Predicted Average AQI : 135.9210
Actual Average AQI    : 138.9167
Average Error         : -2.9957
Absolute Avg Error    : 2.9957
Percentage Error      : 2.16%
Hourly MAE            : 4.9470
Hourly RMSE           : 5.9467
Hourly R²             : 0.5868

------------------------------------------------------------
Day 2
------------------------------------------------------------
Start: 2026-08-02 23:00:00+00:00
End: 2026-08-03 22:00:00+00:00
Hours: 24
Predicted Average AQI : 131.9865
Actual Average AQI    : 129.8750
Average Error         : 2.1115
Absolute Avg Error    : 2.1115
Percentage Error      : 1.63%
Hourly MAE            : 6.9970
Hourly RMSE           : 10.2875
Hourly R²             : 0.6813

------------------------------------------------------------
Day 3
----------------------

In [57]:
# ============================================================
# FINAL DAILY AVERAGE AQI EVALUATION
# ============================================================

print("=" * 60)
print("FINAL DAILY AVERAGE AQI EVALUATION")
print("=" * 60)

final_daily_evaluation = daily_evaluation[
    [
        "forecast_day",
        "forecast_start",
        "forecast_end",
        "hours",
        "predicted_average_aqi",
        "actual_average_aqi",
        "absolute_error",
        "percentage_error"
    ]
].copy()

# Rename for clean presentation
final_daily_evaluation = final_daily_evaluation.rename(
    columns={
        "forecast_day": "Forecast Day",
        "forecast_start": "Start",
        "forecast_end": "End",
        "hours": "Hours",
        "predicted_average_aqi": "Predicted Average AQI",
        "actual_average_aqi": "Actual Average AQI",
        "absolute_error": "Absolute Error",
        "percentage_error": "Percentage Error (%)"
    }
)

# Round values
final_daily_evaluation[
    [
        "Predicted Average AQI",
        "Actual Average AQI",
        "Absolute Error",
        "Percentage Error (%)"
    ]
] = final_daily_evaluation[
    [
        "Predicted Average AQI",
        "Actual Average AQI",
        "Absolute Error",
        "Percentage Error (%)"
    ]
].round(4)

# Display
display(final_daily_evaluation)

print("\n" + "=" * 60)
print("DAY-BY-DAY RESULTS")
print("=" * 60)

for _, row in final_daily_evaluation.iterrows():

    print("\n" + "-" * 60)
    print(f"DAY {int(row['Forecast Day'])}")
    print("-" * 60)

    print(
        f"Predicted Average AQI : "
        f"{row['Predicted Average AQI']:.4f}"
    )

    print(
        f"Actual Average AQI    : "
        f"{row['Actual Average AQI']:.4f}"
    )

    print(
        f"Absolute Error        : "
        f"{row['Absolute Error']:.4f}"
    )

    print(
        f"Percentage Error      : "
        f"{row['Percentage Error (%)']:.2f}%"
    )


# ============================================================
# SAVE FINAL DAILY EVALUATION
# ============================================================

final_daily_evaluation.to_csv(
    "models/evaluation/final_daily_aqi_evaluation.csv",
    index=False
)

print("\n" + "=" * 60)
print("FINAL DAILY EVALUATION SAVED")
print("=" * 60)

print(
    "models/evaluation/final_daily_aqi_evaluation.csv"
)

FINAL DAILY AVERAGE AQI EVALUATION


,Forecast Day,Start,End,Hours,Predicted Average AQI,Actual Average AQI,Absolute Error,Percentage Error (%)
0,1,2026-08-01 23:00:00+00:00,2026-08-02 22:00:00+00:00,24,135.92,138.92,3.00,2.16
1,2,2026-08-02 23:00:00+00:00,2026-08-03 22:00:00+00:00,24,131.99,129.88,2.11,1.63
2,3,2026-08-03 23:00:00+00:00,2026-08-04 22:00:00+00:00,24,151.36,151.58,0.23,0.15



DAY-BY-DAY RESULTS

------------------------------------------------------------
DAY 1
------------------------------------------------------------
Predicted Average AQI : 135.9200
Actual Average AQI    : 138.9200
Absolute Error        : 3.0000
Percentage Error      : 2.16%

------------------------------------------------------------
DAY 2
------------------------------------------------------------
Predicted Average AQI : 131.9900
Actual Average AQI    : 129.8800
Absolute Error        : 2.1100
Percentage Error      : 1.63%

------------------------------------------------------------
DAY 3
------------------------------------------------------------
Predicted Average AQI : 151.3600
Actual Average AQI    : 151.5800
Absolute Error        : 0.2300
Percentage Error      : 0.15%

FINAL DAILY EVALUATION SAVED
models/evaluation/final_daily_aqi_evaluation.csv
